## Hyperparameter tuning

In [ ]:
import itertools

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import itertools

from src.data_loader import load_raw_data
from src.preprocessing import run_preprocessing_pipeline

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, f1_score


def build_and_train(hidden_units, dropout_rate, learning_rate, verbose=0):
    model = keras.Sequential([layers.Input(shape=(splits.X_train.shape[1],))])
    for units in hidden_units:
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )
    model.fit(
        splits.X_train, splits.y_train,
        validation_data=(splits.X_val, splits.y_val),
        epochs=50, batch_size=256, class_weight=class_weight_dict,
        callbacks=[early_stop], verbose=verbose
    )

    y_proba = model.predict(splits.X_val, verbose=0).ravel()
    y_pred = (y_proba >= 0.5).astype(int)
    return model, roc_auc_score(splits.y_val, y_proba), f1_score(splits.y_val, y_pred)


# Search space — small and deliberate, not exhaustive.
architectures = [[32, 16], [64, 32], [128, 64]]
dropout_rates = [0.2, 0.3]
learning_rates = [0.001]

results = []
for hidden_units, dropout_rate, learning_rate in itertools.product(architectures, dropout_rates, learning_rates):
    _, auc, f1 = build_and_train(hidden_units, dropout_rate, learning_rate)
    results.append({
        'hidden_units': str(hidden_units),
        'dropout_rate': dropout_rate,
        'learning_rate': learning_rate,
        'roc_auc': auc,
        'f1': f1,
    })
    print(f"{hidden_units}, dropout={dropout_rate} -> ROC-AUC={auc:.4f}, F1={f1:.4f}")

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
results_df

NameError: name 'keras' is not defined